# Notebook 09: Efficient probability fusion

Use **Google Colab with a CPU runtime**. Run the single code cell. It reads your existing Notebook 07 manifest and Notebook 02/04/05 action caches automatically. **Notebook 08 is not required.** Keep those earlier Drive outputs in place. No dataset upload or extra installation is required in a standard Colab runtime.

This experiment changes the method: a small gate combines the keep, mask and reconstruction probabilities instead of choosing a single candidate. A learned fixed-weight mixture, uniform mixture and regularized residual stacking model provide controls. All signal models remain frozen. The three fitted meta-models use only the policy partition. Each model trains for 50 fixed epochs; blending and decision margins are selected on validation only. Neither stuck faults nor test labels enter fitting or selection.

Two operating points are reported for every family:

- **Safe:** original empirical validation constraints (clean Macro-F1 loss at most 0.5 pp; clean and mean development-fault harm at most 1%). These are not test risk guarantees.
- **Accuracy:** retains the clean Macro-F1 constraint, but does not constrain harm. Always report its harm together with its accuracy. It is not a comparison under the safe profile's constraints.

**No improvement is guaranteed.** This is an exploratory extension after prior test inspection. Fusion is an established technique; this notebook does not establish novelty, significance or state-of-the-art performance. All original methods and both operating points remain in the output. Do not select a test winner and call that selection confirmatory.

The old label-assisted oracle chooses one candidate. It is a diagnostic for that action set, not an upper bound for probability fusion or stacking. The new decision-change rate counts changed predicted classes; it is not the old sensor-action intervention rate.

The cell checkpoints every ten epochs and after each configuration. It pauses at a 75-minute session budget. If it prints **PAUSED**, rerun the same cell; completed work is reused. A disconnect can lose at most the training since the last checkpoint. The budget is not an estimate of total runtime. Outputs are saved in `SensorRepair_ICDSIS/outputs/notebook09_v1`.

Send the final printed output and `notebook09_aggregate_summary.csv`, `notebook09_comparisons.csv`, and `notebook09_run_report.json` when complete. No synthetic example outputs are included.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import gc
import hashlib
import json
import os
import random
import shutil
import time
import numpy as np
import pandas as pd
import torch
from torch import nn

PROJECT=Path('/content/drive/MyDrive/SensorRepair_ICDSIS')
OUTPUT=PROJECT/'outputs'/'notebook09_v1'
LOCAL=Path('/content/sensorrepair_notebook09_v1')
CODE_VERSION='sensorrepair_nb09_v1'
CELL_SIGNATURE='b7febceca1071fa9ca5a5c5b2dec6ec4bae52cdf04a9bf7dbc426e53155d5c3c'
EPOCHS=50
BATCH_SIZE=512
LEARNING_RATE=.003
WEIGHT_DECAY=.01
HIDDEN=32
PRESERVATION_WEIGHT=.25
REPAIR_MASS_WEIGHT=.005
SESSION_MINUTES=75
ALPHAS=[0.,.25,.5,.75,1.]
MARGINS=[0.,.025,.05,.1]
CLEAN_F1_TOLERANCE=.005
MAX_SAFE_HARM=.01
FAMILIES=['uniform_fusion','global_fusion','gated_fusion','residual_stacking']
PROFILES=['safe','accuracy']
NEW_METHODS=[family+'_'+profile for family in FAMILIES for profile in PROFILES]
OLD_METHODS=['keep','mask_suspect','reconstruct_suspect','residual_gate_mask','residual_gate_reconstruct',
             'confidence_select','oracle_diagnostic','harm_aware_policy','benefit_only_policy',
             'confidence_margin_tuned','residual_mask_tuned','residual_reconstruct_tuned',
             'ternary_outcome_selector','utility_regression_selector']
METHODS=OLD_METHODS+NEW_METHODS
CONDITIONS=['clean']+[f'{f}_{i}' for f in ['dropout','bias','gain','stuck'] for i in [1,2,3]]
SCOPES={'clean':['clean'],'known_faults':CONDITIONS[1:10],'unseen_stuck':CONDITIONS[10:]}
METRICS=['macro_f1','accuracy','decision_change_rate','helpful_rate','harmful_rate']
OUTPUT.mkdir(parents=True,exist_ok=True)
LOCAL.mkdir(parents=True,exist_ok=True)
torch.set_num_threads(2)
torch.use_deterministic_algorithms(True)
DEADLINE=time.monotonic()+SESSION_MINUTES*60
print('NOTEBOOK 09 | conservative probability fusion | CPU | one executable cell',flush=True)
print('Both datasets; 15 configurations; cached signal-model predictions reused.',flush=True)
print('Three small meta-models per configuration. No CNN/reconstructor training or GPU needed.',flush=True)
print('Exploratory extension: existing test results are already known.',flush=True)
print('Safe profile retains the original validation harm guards. Accuracy profile has no harm guard.',flush=True)


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def json_signature(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, allow_nan=False).encode()).hexdigest()


def save_json(path, value):
    path = Path(path)
    temporary = path.with_name(path.name + '.partial')
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False), encoding='utf-8')
    os.replace(temporary, path)


def save_csv(path, value):
    path = Path(path)
    temporary = path.with_name(path.name + '.partial')
    value.to_csv(temporary, index=False)
    os.replace(temporary, path)


def save_npz(path, **arrays):
    path = Path(path)
    temporary = path.with_name(path.name + '.partial')
    with temporary.open('wb') as stream:
        np.savez_compressed(stream, **arrays)
    os.replace(temporary, path)


def probability_statistics(probability):
    sorted_p = np.sort(probability, axis=-1)
    confidence = sorted_p[..., -1]
    margin = confidence - sorted_p[..., -2]
    entropy = -(probability * np.log(np.clip(probability, 1e-7, 1.0))).sum(axis=-1)
    return np.stack([confidence, margin, entropy], axis=-1)


def policy_features(probabilities, residuals, statistics, residual_scale):
    # This function accepts observable arrays only. It cannot access labels, subjects,
    # source IDs, injected fault locations, or condition/severity metadata.
    D, N, A, K = probabilities.shape
    G, J, M = residuals.shape[-1], A - 1, D * N
    p = probabilities.reshape(M, A, K)
    r = residuals.reshape(M, G)
    s = statistics.reshape(M, G, 4)
    original, candidates = p[:, 0], p[:, 1:]
    candidate_groups = np.tile(np.arange(G), 2)
    original_statistics = probability_statistics(original)
    candidate_statistics = probability_statistics(candidates)
    scaled = r / residual_scale[None, :]

    def repeat(values):
        return np.broadcast_to(values[:, None, :], (M, J, values.shape[-1]))

    middle = (original[:, None, :] + candidates) / 2.0
    js = 0.5 * (original[:, None, :] * np.log(np.clip(original[:, None, :], 1e-7, 1.0) / np.clip(middle, 1e-7, 1.0))).sum(axis=-1)
    js += 0.5 * (candidates * np.log(np.clip(candidates, 1e-7, 1.0) / np.clip(middle, 1e-7, 1.0))).sum(axis=-1)
    names, pieces = [], []

    def add(values, labels):
        assert values.shape == (M, J, len(labels))
        pieces.append(values)
        names.extend(labels)

    add(repeat(original), [f'original_probability_{k}' for k in range(K)])
    add(candidates, [f'action_probability_{k}' for k in range(K)])
    add(candidates - original[:, None, :], [f'probability_difference_{k}' for k in range(K)])
    add(repeat(original_statistics), ['original_confidence', 'original_margin', 'original_entropy'])
    add(candidate_statistics, ['action_confidence', 'action_margin', 'action_entropy'])
    add(candidate_statistics - original_statistics[:, None, :], ['confidence_difference', 'margin_difference', 'entropy_difference'])
    add(js[..., None], ['prediction_js_divergence'])
    add((candidates.argmax(axis=-1) == original.argmax(axis=-1)[:, None])[..., None], ['prediction_agreement'])
    kinds = np.zeros((J, 2), dtype=np.float32)
    kinds[:G, 0], kinds[G:, 1] = 1.0, 1.0
    add(np.broadcast_to(kinds[None], (M, J, 2)), ['action_is_mask', 'action_is_reconstruction'])
    add(np.broadcast_to(np.eye(G, dtype=np.float32)[candidate_groups][None], (M, J, G)), [f'candidate_group_{g}' for g in range(G)])
    add(repeat(np.log1p(r)), [f'log_residual_group_{g}' for g in range(G)])
    add(np.log1p(r[:, candidate_groups])[..., None], ['candidate_log_residual'])
    add(np.log1p(scaled[:, candidate_groups])[..., None], ['candidate_log_scaled_residual'])
    add(repeat(np.log1p(scaled.max(axis=-1, keepdims=True))), ['maximum_log_scaled_residual'])
    add(s[:, candidate_groups, :], ['candidate_mean_absolute_signal', 'candidate_signal_std', 'candidate_temporal_difference', 'candidate_zero_fraction'])
    add(repeat(s.reshape(M, G * 4)), [f'observed_group_{g}_stat_{j}' for g in range(G) for j in range(4)])
    values = np.ascontiguousarray(np.concatenate(pieces, axis=-1).reshape(M * J, -1), dtype=np.float32)
    assert len(set(names)) == len(names) and np.isfinite(values).all()
    return values, names


def macro_f1(y, prediction, classes):
    matrix = np.bincount(y * classes + prediction, minlength=classes * classes).reshape(classes, classes)
    numerator = 2.0 * matrix.diagonal()
    denominator = matrix.sum(axis=0) + matrix.sum(axis=1)
    return float(np.divide(numerator, denominator, out=np.zeros(classes), where=denominator > 0).mean())


def pack_from_caches(paths, expected_signature):
    matrices, residuals, statistics = [], [], []
    anchor, conditions = None, []
    for path in paths:
        with np.load(path, allow_pickle=False) as saved:
            assert saved['signature'].item() == expected_signature
            identity = {key: saved[key] for key in ['y', 'subjects', 'window_ids', 'source_indices', 'action_names']}
            if anchor is None:
                anchor = identity
            else:
                for key, value in identity.items():
                    assert np.array_equal(value, anchor[key]), f'Cache alignment error: {key}'
            matrices.append(saved['probabilities'])
            residuals.append(saved['reconstruction_residuals'])
            statistics.append(saved['observed_group_statistics'])
            conditions.append(str(saved['condition'].item()))
    pack = dict(anchor)
    pack.update(probabilities=np.stack(matrices), residuals=np.stack(residuals),
                statistics=np.stack(statistics), conditions=conditions)
    p, r, s = pack['probabilities'], pack['residuals'], pack['statistics']
    D, N, A, K = p.shape
    G = r.shape[-1]
    assert A == 1 + 2 * G and r.shape == (D, N, G) and s.shape == (D, N, G, 4)
    assert np.isfinite(p).all() and np.isfinite(r).all() and np.isfinite(s).all()
    assert np.all(p >= 0) and np.allclose(p.sum(axis=-1), 1.0, atol=1e-5)
    assert np.all(r >= 0) and len(np.unique(pack['window_ids'])) == N
    return pack


def assert_table_parity(computed,original,keys,columns):
    assert not computed.duplicated(keys).any() and not original.duplicated(keys).any()
    a=computed.set_index(keys).sort_index()
    b=original.set_index(keys).sort_index()
    assert a.index.equals(b.index),'Table row identities changed.'
    for col in columns:
        assert np.allclose(a[col].to_numpy(dtype=float),b[col].to_numpy(dtype=float),atol=1e-12,rtol=0,equal_nan=True),f'Metric parity failed: {col}'

class PauseStudy(Exception):
    pass


def check_budget():
    if time.monotonic()>=DEADLINE:raise PauseStudy('Session budget reached. Rerun the same cell.')


def identity(job):
    return f'{job["dataset"]}_{job["fold"]}_seed_{job["seed"]}'


def verified_copy(source,relative,expected):
    target=LOCAL/'inputs'/relative
    target.parent.mkdir(parents=True,exist_ok=True)
    temporary=target.with_name(target.name+'.partial')
    shutil.copyfile(source,temporary)
    assert file_sha256(temporary)==expected,f'Input hash mismatch: {source}'
    os.replace(temporary,target)
    return target


SOURCE=PROJECT/'outputs/notebook07_v1'
records=json.loads((SOURCE/'notebook07_artifact_inventory.json').read_text())
inventory={r['filename']:r for r in records}
assert len(records)==len(inventory)


def root_file(name):
    return verified_copy(SOURCE/name,'nb07/'+name,inventory[name]['sha256'])


source_report=json.loads(root_file('notebook07_run_report.json').read_text())
source_manifest=json.loads(root_file('notebook07_study_manifest.json').read_text())
assert source_report['study_complete'] and source_report['configurations_complete']==15
assert source_report['study_signature']==source_manifest['signature']==json_signature(source_manifest['plan'])
RESULTS=source_report['completed_results']
assert len({identity(r) for r in RESULTS})==len(RESULTS)==15
EXPECTED={(d,f'fold_{f}',s) for s in [2026,2027,2028] for d in ['dsads','har']
          for f in (range(4) if d=='dsads' else range(1))}
assert {(r['dataset'],r['fold'],r['seed']) for r in RESULTS}==EXPECTED
CACHE_INPUTS=source_manifest['plan']['cache_inputs']
STAGES={f'nb{i:02d}':PROJECT/'outputs'/f'notebook{i:02d}_v1' for i in [2,4,5]}
old_aggregate=pd.read_csv(root_file('notebook07_aggregate_summary.csv'))
plan={'code_version':CODE_VERSION,'cell_signature':CELL_SIGNATURE,'source_study_signature':source_report['study_signature'],
      'source_report_sha256':inventory['notebook07_run_report.json']['sha256'],
      'cache_inputs':CACHE_INPUTS,'jobs':[{k:r[k] for k in ['dataset','fold','seed']} for r in RESULTS],
      'fit_partition':'policy only, clean and nine development conditions',
      'selection_partition':'validation only, clean and nine development conditions',
      'test_status':'exploratory extension after inspecting previous test outcomes',
      'primary_new_method':'gated_fusion_safe','accuracy_profile':'secondary operating point with no harm guard',
      'model_definitions':{'global_fusion':'one shared learned softmax vector over cached candidate probabilities',
          'gated_fusion':'32-unit shared MLP maps observable original action features to sample-specific mixture weights',
          'residual_stacking':'linear bounded logit correction to temperature-2 keep probabilities, using cached probabilities/residuals/statistics',
          'uniform_fusion':'untrained equal-weight probability mixture'},
      'training':{'epochs':EPOCHS,'batch_size':BATCH_SIZE,'lr':LEARNING_RATE,'weight_decay':WEIGHT_DECAY,
          'hidden':HIDDEN,'dropout':.1,'preservation_weight':PRESERVATION_WEIGHT,'repair_mass_weight':REPAIR_MASS_WEIGHT,
          'loss':'class-balanced log loss plus originally-correct probability-preservation penalty; mixture models also penalize repair mass',
          'checkpoint_selection':'none; fixed final epoch; checkpoints only for resuming'},
      'validation_grid':{'alpha':ALPHAS,'margin':MARGINS},
      'validation_guards':{'both_profiles_clean_f1_loss':CLEAN_F1_TOLERANCE,'safe_clean_and_mean_corrupt_harm':MAX_SAFE_HARM,
          'accuracy_harm_guard':None},
      'decision_change_rate':'fraction with a different predicted class than keep; distinct from original sensor action intervention rate',
      'old_oracle_interpretation':'label-assisted single-candidate diagnostic; not an upper bound for probability fusion or stacking',
      'aggregation':source_report['aggregation'],'gpu_required':False,
      'software':{'torch':str(torch.__version__),'numpy':str(np.__version__),'pandas':str(pd.__version__)},
      'novelty_or_significance_guarantee':False,
      'references':['https://docs.pytorch.org/docs/stable/generated/torch.optim.AdamW.html',
                    'https://docs.pytorch.org/docs/stable/notes/randomness.html',
                    'https://doi.org/10.1109/TKDE.2023.3240858']}
STUDY_SIGNATURE=json_signature(plan)
manifest_path=OUTPUT/'notebook09_study_manifest.json'
if manifest_path.exists():
    assert json.loads(manifest_path.read_text())['signature']==STUDY_SIGNATURE,'Study inputs, code, or environment changed. Keep the original configuration.'
else:save_json(manifest_path,{'signature':STUDY_SIGNATURE,'plan':plan})
print('Exploratory study plan frozen:',STUDY_SIGNATURE[:16],flush=True)


def load_pack(result,split):
    assert split in ['policy','validation','test']
    paths=[];expected=None
    for condition in (CONDITIONS if split=='test' else CONDITIONS[:10]):
        r=CACHE_INPUTS[identity(result)][split+'|'+condition]
        path=verified_copy(STAGES[r['stage']]/r['filename'],r['stage']+'/'+r['filename'],r['sha256'])
        if expected is None:expected=r['signature']
        assert expected==r['signature']
        with np.load(path,allow_pickle=False) as z:
            assert z['split'].item()==split and z['condition'].item()==condition
        paths.append(path)
    pack=pack_from_caches(paths,expected)
    assert pack['conditions']==(CONDITIONS if split=='test' else CONDITIONS[:10])
    assert np.array_equal(np.unique(pack['y']),np.arange(pack['probabilities'].shape[-1]))
    return pack


def raw_features(probabilities,residuals,statistics,scale):
    # Runtime features take observable arrays only. No labels or split/fault metadata.
    D,N,A,K=probabilities.shape
    X,names=policy_features(probabilities,residuals,statistics,scale)
    gate=X.reshape(D*N,A-1,-1)
    stack=np.concatenate([probabilities.reshape(D*N,A*K),np.log1p(residuals.reshape(D*N,-1)/scale),
                          statistics.reshape(D*N,-1)],axis=1).astype(np.float32)
    assert np.isfinite(gate).all() and np.isfinite(stack).all()
    return gate,stack,names


def prepare_inputs(pack,scale,normalization=None):
    gate,stack,names=raw_features(pack['probabilities'],pack['residuals'],pack['statistics'],scale)
    if normalization is None:
        normalization={'gate_mean':gate.mean(axis=(0,1),dtype=np.float64).astype(np.float32),
                       'gate_std':np.maximum(gate.std(axis=(0,1),dtype=np.float64),1e-4).astype(np.float32),
                       'stack_mean':stack.mean(axis=0,dtype=np.float64).astype(np.float32),
                       'stack_std':np.maximum(stack.std(axis=0,dtype=np.float64),1e-4).astype(np.float32)}
    gate=np.clip((gate-normalization['gate_mean'])/normalization['gate_std'],-8,8)
    stack=np.clip((stack-normalization['stack_mean'])/normalization['stack_std'],-8,8)
    p=pack['probabilities'].reshape(-1,*pack['probabilities'].shape[2:])
    return {'p':torch.from_numpy(np.ascontiguousarray(p,dtype=np.float32)),
            'gate':torch.from_numpy(np.ascontiguousarray(gate,dtype=np.float32)),
            'stack':torch.from_numpy(np.ascontiguousarray(stack,dtype=np.float32))},normalization,names


class FusionModel(nn.Module):
    def __init__(self,kind,A,K,F,S):
        super().__init__();self.kind=kind
        if kind=='global_fusion':
            self.action_logits=nn.Parameter(torch.tensor([0.]+[-3.]*(A-1)))
        elif kind=='gated_fusion':
            self.gate=nn.Sequential(nn.Linear(F,HIDDEN),nn.SiLU(),nn.Dropout(.1),nn.Linear(HIDDEN,1))
            nn.init.zeros_(self.gate[-1].weight);nn.init.zeros_(self.gate[-1].bias)
            self.action_bias=nn.Parameter(torch.full((A-1,),-3.))
        elif kind=='residual_stacking':
            self.correction=nn.Linear(S,K)
            nn.init.zeros_(self.correction.weight);nn.init.zeros_(self.correction.bias)
        else:raise ValueError(kind)

    def forward(self,p,gate,stack):
        if self.kind=='residual_stacking':
            q=torch.softmax(torch.log(p[:,0].clamp_min(1e-7))/2.+2.*torch.tanh(self.correction(stack)),dim=-1)
            return q,None
        if self.kind=='global_fusion':
            logits=self.action_logits[None].expand(len(p),-1)
        else:
            logits=torch.cat([torch.zeros((len(p),1)),self.gate(gate).squeeze(-1)+self.action_bias],dim=1)
        weights=torch.softmax(logits,dim=1)
        return (p*weights[:,:,None]).sum(dim=1),weights


def checkpoint_save(path,payload):
    temporary=LOCAL/(path.parent.name+'_'+path.name+'.tmp')
    torch.save(payload,temporary)
    partial=path.with_name(path.name+'.partial')
    shutil.copyfile(temporary,partial);os.replace(partial,path)


def fit_model(kind,inputs,y,job,run,run_signature):
    offset={'global_fusion':110,'gated_fusion':220,'residual_stacking':330}[kind]
    model_seed=int(job['seed'])+offset
    random.seed(model_seed);np.random.seed(model_seed);torch.manual_seed(model_seed)
    p=inputs['p'];A,K=p.shape[1:]
    dimensions={'A':A,'K':K,'F':inputs['gate'].shape[-1],'S':inputs['stack'].shape[-1]}
    model=FusionModel(kind,**dimensions)
    optimizer=torch.optim.AdamW(model.parameters(),lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)
    target=torch.tensor(np.tile(y,int(len(p)/len(y))),dtype=torch.long)
    assert len(target)==len(p)
    counts=torch.bincount(target,minlength=K).float()
    assert torch.all(counts>0)
    class_weight=(counts.sum()/(K*counts)).clamp(.25,4.)
    path=run/(kind+'.pt');start=0;seconds=0.;loss_value=None
    if path.exists():
        saved=torch.load(path,map_location='cpu',weights_only=False)
        assert saved['signature']==run_signature and saved['kind']==kind and saved['dimensions']==dimensions
        model.load_state_dict(saved['model']);optimizer.load_state_dict(saved['optimizer'])
        torch.set_rng_state(saved['torch_rng']);start=saved['epoch'];seconds=saved['training_seconds'];loss_value=saved['loss']
    def store(epoch):
        checkpoint_save(path,{'signature':run_signature,'kind':kind,'dimensions':dimensions,
            'model':model.state_dict(),'optimizer':optimizer.state_dict(),'torch_rng':torch.get_rng_state(),
            'epoch':epoch,'training_seconds':seconds,'loss':loss_value})
    if start>=EPOCHS:
        print(f'  Reused {kind}: {start} epochs.',flush=True)
    for epoch in range(start+1,EPOCHS+1):
        if time.monotonic()>=DEADLINE:
            store(epoch-1);raise PauseStudy('Meta-model checkpoint saved. Rerun this same cell.')
        model.train();began=time.perf_counter();total=0.;number=0
        order=np.random.default_rng(model_seed+epoch*1009).permutation(len(p))
        for first in range(0,len(p),BATCH_SIZE):
            use=torch.from_numpy(order[first:first+BATCH_SIZE])
            b={k:v[use] for k,v in inputs.items()};labels=target[use]
            q,weights=model(b['p'],b['gate'],b['stack'])
            logq=torch.log(q.clamp_min(1e-7));logp=torch.log(b['p'][:,0].clamp_min(1e-7))
            true_logq=logq.gather(1,labels[:,None]).squeeze(1)
            true_logp=logp.gather(1,labels[:,None]).squeeze(1)
            correct=b['p'][:,0].argmax(1)==labels
            ce=(-true_logq*class_weight[labels]).mean()
            preservation=(torch.relu(true_logp-true_logq)*correct).mean()
            loss=ce+PRESERVATION_WEIGHT*preservation
            if weights is not None:loss=loss+REPAIR_MASS_WEIGHT*(1.-weights[:,0]).mean()
            assert torch.isfinite(loss),'Non-finite meta-model loss.'
            optimizer.zero_grad(set_to_none=True);loss.backward();nn.utils.clip_grad_norm_(model.parameters(),5.);optimizer.step()
            total+=float(loss.detach())*len(use);number+=len(use)
        seconds+=time.perf_counter()-began;loss_value=total/number
        if epoch%10==0 or epoch==EPOCHS:store(epoch)
        if epoch in [1,10,30,EPOCHS]:print(f'  {kind} epoch {epoch:02d}/{EPOCHS} | policy loss {loss_value:.4f}',flush=True)
    cost={**job,'model':kind,'parameters':sum(t.numel() for t in model.parameters()),'epochs':EPOCHS,
          'policy_views':len(target),'unique_policy_windows':len(y),'training_seconds':seconds,'checkpoint_bytes':path.stat().st_size}
    return model,cost


def predict_model(model,inputs):
    model.eval();chunks=[];weight_chunks=[]
    with torch.inference_mode():
        for start in range(0,len(inputs['p']),1024):
            b={k:v[start:start+1024] for k,v in inputs.items()}
            q,weights=model(b['p'],b['gate'],b['stack']);chunks.append(q.numpy())
            if weights is not None:weight_chunks.append(weights.numpy())
    q=np.concatenate(chunks)
    assert np.isfinite(q).all() and np.all(q>=0) and np.allclose(q.sum(1),1.,atol=1e-5)
    return q,(np.concatenate(weight_chunks) if weight_chunks else None)


def blended_predictions(p0,fused,alpha,margin):
    q=(1.-alpha)*p0+alpha*fused
    base=p0.argmax(-1);proposal=q.argmax(-1)
    gap=q.max(-1)-np.take_along_axis(q,base[...,None],axis=-1).squeeze(-1)
    return np.where((proposal!=base)&(gap>margin),proposal,base).astype(np.int64)


def evaluate_predictions(pack,prediction,method,job):
    y=pack['y'];K=pack['probabilities'].shape[-1];base=pack['probabilities'][:,:,0].argmax(-1)
    assert prediction.shape==base.shape
    rows=[]
    for d,condition in enumerate(pack['conditions']):
        correct=prediction[d]==y;was_correct=base[d]==y
        helpful=float((~was_correct&correct).mean());harmful=float((was_correct&~correct).mean())
        assert abs(correct.mean()-was_correct.mean()-helpful+harmful)<1e-10
        rows.append({**job,'method':method,'condition':condition,'windows':len(y),
            'macro_f1':macro_f1(y,prediction[d],K),'accuracy':float(correct.mean()),
            'decision_change_rate':float((prediction[d]!=base[d]).mean()),'helpful_rate':helpful,'harmful_rate':harmful})
    return rows


def validation_summary(rows):
    clean=rows[0];faults=rows[1:]
    return {**{m:float(np.mean([r[m] for r in faults])) for m in METRICS},
            'clean_macro_f1':clean['macro_f1'],'clean_harmful_rate':clean['harmful_rate']}


def select_profiles(validation,probabilities,job):
    selections={};trials=[];p0=validation['probabilities'][:,:,0]
    keep_score=validation_summary(evaluate_predictions(validation,p0.argmax(-1),'keep',job))
    for family,q in probabilities.items():
        best={profile:None for profile in PROFILES}
        for alpha in ALPHAS:
            for margin in (MARGINS if alpha>0 else [0.]):
                prediction=blended_predictions(p0,q,alpha,margin)
                metrics=validation_summary(evaluate_predictions(validation,prediction,family,job))
                clean_ok=metrics['clean_macro_f1']>=keep_score['clean_macro_f1']-CLEAN_F1_TOLERANCE-1e-12
                safe_ok=clean_ok and metrics['clean_harmful_rate']<=MAX_SAFE_HARM+1e-12 and metrics['harmful_rate']<=MAX_SAFE_HARM+1e-12
                for profile,feasible in [('safe',safe_ok),('accuracy',clean_ok)]:
                    settings={'family':family,'profile':profile,'alpha':alpha,'margin':margin}
                    trials.append({**settings,'feasible':bool(feasible),**metrics})
                    rank=(metrics['macro_f1'],metrics['clean_macro_f1'],-metrics['harmful_rate'],-metrics['decision_change_rate'],-alpha)
                    if feasible and (best[profile] is None or rank>best[profile]['rank']):
                        best[profile]={'rank':rank,'settings':settings,'summary':metrics,
                                       'structural_keep_only':alpha==0.,'keep_only_on_validation':bool(np.all(prediction==p0.argmax(-1)))}
        for profile,item in best.items():
            assert item is not None,'Keep fallback must remain feasible.'
            selections[family+'_'+profile]={k:v for k,v in item.items() if k!='rank'}
    return selections,trials


def summarize(rows):
    frame=pd.DataFrame(rows);out=[]
    for method in frame.method.unique():
        for scope,conditions in SCOPES.items():
            part=frame[(frame.method==method)&frame.condition.isin(conditions)]
            assert len(part)==len(conditions)
            out.append({**{k:part.iloc[0][k] for k in ['dataset','fold','seed']},'method':method,'scope':scope,
                        **{m:float(part[m].mean()) for m in METRICS}})
    return out


def run_file(result,name):
    item=next(r for r in result['artifacts'] if r['filename']==name)
    relative='runs/'+identity(result)+'/'+name
    return verified_copy(SOURCE/relative,'nb07/'+relative,item['sha256'])


def evaluate_job(result):
    job={k:result[k] for k in ['dataset','fold','seed']};name=identity(job)
    run=OUTPUT/'runs'/name;run.mkdir(parents=True,exist_ok=True)
    run_signature=json_signature({'study':STUDY_SIGNATURE,**job})
    policy,validation=load_pack(result,'policy'),load_pack(result,'validation')
    for field in ['subjects','window_ids']:
        assert not set(policy[field].tolist())&set(validation[field].tolist()),'Policy/validation overlap.'
    assert policy['action_names'].tolist()==validation['action_names'].tolist()
    scale=np.maximum(np.quantile(policy['residuals'][0],.95,axis=0),1e-8)
    policy_inputs,normalization,feature_names=prepare_inputs(policy,scale)
    assert len(feature_names)==result['feature_count']
    val_inputs,_,val_names=prepare_inputs(validation,scale,normalization)
    assert val_names==feature_names
    models={};costs=[]
    for kind in FAMILIES[1:]:
        check_budget();models[kind],cost=fit_model(kind,policy_inputs,policy['y'],job,run,run_signature);costs.append(cost)
    shape=validation['probabilities'].shape
    val_probs={'uniform_fusion':validation['probabilities'].mean(axis=2)}
    for kind,model in models.items():val_probs[kind]=predict_model(model,val_inputs)[0].reshape(shape[0],shape[1],shape[-1])
    selections,trials=select_profiles(validation,val_probs,job)
    selected={'signature':run_signature,'selection_split':'validation','selected_methods':selections,
        'policy_participants':np.unique(policy['subjects']).tolist(),'validation_participants':np.unique(validation['subjects']).tolist(),
        'validation_window_ids':validation['window_ids'].tolist(),
        'feature_names':feature_names,'normalization_fit':'policy only',
        'excluded_runtime_features':['y','subjects','window_ids','source_indices','condition','severity','injected_fault_location'],
        'status':'exploratory; prior test results already inspected'}
    selected_path=run/'selected_fusion.json'
    if selected_path.exists():assert json.loads(selected_path.read_text())==selected,'Frozen validation selection changed.'
    else:save_json(selected_path,selected)
    save_csv(run/'validation_trials.csv',pd.DataFrame(trials))
    save_npz(run/'feature_normalization.npz',residual_scale=scale,**normalization)
    del policy_inputs,val_inputs,validation,val_probs
    gc.collect();check_budget()
    # First access to test arrays for this run is after the selections above are committed.
    test=load_pack(result,'test')
    for field in ['subjects','window_ids']:
        other=selected['validation_participants'] if field=='subjects' else selected['validation_window_ids']
        assert not set(test[field].tolist())&(set(policy[field].tolist())|set(other)), 'Test overlap.'
    assert test['action_names'].tolist()==policy['action_names'].tolist()
    with np.load(run_file(result,'test_predictions.npz'),allow_pickle=False) as old:
        assert old['signature'].item()==result['signature']
        assert old['methods'].tolist()==OLD_METHODS and old['conditions'].tolist()==CONDITIONS
        for key in ['y','subjects','window_ids','source_indices']:assert np.array_equal(old[key],test[key])
        predictions={m:old['predictions'][i].copy() for i,m in enumerate(OLD_METHODS)}
        old_choices=old['choices']
    candidates=test['probabilities'].argmax(-1)
    for i,method in enumerate(OLD_METHODS):
        assert np.array_equal(np.take_along_axis(candidates,old_choices[i,...,None],axis=2).squeeze(2),predictions[method])
    test_inputs,_,test_names=prepare_inputs(test,scale,normalization)
    assert test_names==feature_names
    D,N,A,K=test['probabilities'].shape
    fused={'uniform_fusion':test['probabilities'].mean(2)};weight_rows=[]
    for kind,model in models.items():
        q,weights=predict_model(model,test_inputs);fused[kind]=q.reshape(D,N,K)
        if weights is not None:
            weights=weights.reshape(D,N,A);G=(A-1)//2
            for d,condition in enumerate(CONDITIONS):
                weight_rows.append({**job,'family':kind,'condition':condition,'keep_weight':float(weights[d,:,0].mean()),
                    'mask_weight':float(weights[d,:,1:G+1].sum(1).mean()),'reconstruction_weight':float(weights[d,:,G+1:].sum(1).mean())})
    for method,item in selections.items():
        s=item['settings'];predictions[method]=blended_predictions(test['probabilities'][:,:,0],fused[s['family']],s['alpha'],s['margin'])
        print(f'  {method}: alpha={s["alpha"]}, margin={s["margin"]} (validation selected)',flush=True)
    rows=[r for method,pred in predictions.items() for r in evaluate_predictions(test,pred,method,job)]
    summary=summarize(rows)
    old_summary=pd.read_csv(run_file(result,'test_summary.csv'))
    assert_table_parity(pd.DataFrame(summary).query('method in @OLD_METHODS'),old_summary,
                       ['dataset','fold','seed','method','scope'],['macro_f1','accuracy','helpful_rate','harmful_rate'])
    save_npz(run/'test_predictions.npz',signature=np.asarray(run_signature),methods=np.asarray(METHODS),conditions=np.asarray(CONDITIONS),
        predictions=np.stack([predictions[m] for m in METHODS]),fusion_families=np.asarray(FAMILIES),
        fusion_probabilities=np.stack([fused[m] for m in FAMILIES]),
        **{key:test[key] for key in ['y','subjects','window_ids','source_indices']})
    for filename,values in [('per_condition',rows),('test_summary',summary),('model_costs',costs),('mixture_weights',weight_rows)]:
        save_csv(run/(filename+'.csv'),pd.DataFrame(values))
    output={**job,'signature':run_signature,'original_prediction_parity':True,'original_summary_parity':True,
            'selected_methods':selections,'policy_windows':len(policy['y']),'test_windows':N,'fitted_meta_models':3,
            'artifacts':[{'filename':p.name,'sha256':file_sha256(p)} for p in
                [run/'test_predictions.npz',run/'selected_fusion.json',run/'per_condition.csv',run/'test_summary.csv',
                 run/'model_costs.csv',run/'mixture_weights.csv',run/'feature_normalization.npz']+[run/(k+'.pt') for k in FAMILIES[1:]]]}
    save_json(run/'dataset_complete.json',output)
    return output


def completed(result):
    job={k:result[k] for k in ['dataset','fold','seed']}
    run=OUTPUT/'runs'/identity(job);path=run/'dataset_complete.json'
    if not path.exists():return None
    data=json.loads(path.read_text())
    assert data['signature']==json_signature({'study':STUDY_SIGNATURE,**job})
    for item in data['artifacts']:assert file_sha256(run/item['filename'])==item['sha256'],'Completed output changed.'
    return data


def publish(final=False):
    done=[x for r in RESULTS if (x:=completed(r)) is not None]
    if not done:
        print('NOTEBOOK 09 PAUSED. Models/checkpoints are saved; rerun this cell.',flush=True);return
    tables={}
    for name in ['test_summary','per_condition','model_costs','mixture_weights']:
        tables[name]=pd.concat([pd.read_csv(OUTPUT/'runs'/identity(r)/(name+'.csv')) for r in done],ignore_index=True)
        save_csv(OUTPUT/('notebook09_'+name+'.csv'),tables[name])
    seeds=[]
    for keys,part in tables['test_summary'].groupby(['dataset','seed','method','scope'],sort=False):
        dataset,seed,method,scope=keys;required=4 if dataset=='dsads' else 1
        assert len(part)==part.fold.nunique()<=required
        if len(part)==required:seeds.append({'dataset':dataset,'seed':int(seed),'method':method,'scope':scope,
                                            **{m:float(part[m].mean()) for m in METRICS}})
    seed_frame=pd.DataFrame(seeds,columns=['dataset','seed','method','scope']+METRICS)
    save_csv(OUTPUT/'notebook09_per_seed_summary.csv',seed_frame)
    aggregated=[]
    for keys,part in seed_frame.groupby(['dataset','method','scope'],sort=False):
        item=dict(zip(['dataset','method','scope'],keys));item['complete_seeds']=len(part)
        for m in METRICS:
            item[m+'_mean']=float(part[m].mean());item[m+'_seed_sd']=float(part[m].std(ddof=1)) if len(part)>1 else None
        aggregated.append(item)
    aggregate=pd.DataFrame(aggregated,columns=['dataset','method','scope','complete_seeds']+[m+s for m in METRICS for s in ['_mean','_seed_sd']])
    save_csv(OUTPUT/'notebook09_aggregate_summary.csv',aggregate)
    ready=len(done)==15;comparisons=[]
    if ready:
        assert_table_parity(aggregate[aggregate.method.isin(OLD_METHODS)],old_aggregate,['dataset','method','scope'],
            [m+s for m in ['macro_f1','accuracy','helpful_rate','harmful_rate'] for s in ['_mean','_seed_sd']])
    if len(aggregate):
        lookup=aggregate.set_index(['dataset','method','scope'])
        for dataset in aggregate.dataset.unique():
            for scope in SCOPES:
                for profile in PROFILES:
                    proposed=lookup.loc[(dataset,'gated_fusion_'+profile,scope)]
                    controls=['keep','harm_aware_policy','confidence_select','ternary_outcome_selector','utility_regression_selector',
                              'uniform_fusion_'+profile,'global_fusion_'+profile,'residual_stacking_'+profile]
                    for comparator in controls:
                        control=lookup.loc[(dataset,comparator,scope)]
                        comparisons.append({'dataset':dataset,'scope':scope,'profile':profile,'comparator':comparator,
                            'complete_seeds':int(proposed.complete_seeds),'proposed_macro_f1_pct':100*proposed.macro_f1_mean,
                            'comparator_macro_f1_pct':100*control.macro_f1_mean,
                            'delta_macro_f1_pp':100*(proposed.macro_f1_mean-control.macro_f1_mean),
                            'proposed_harm_pct':100*proposed.harmful_rate_mean,'comparator_harm_pct':100*control.harmful_rate_mean,
                            'proposed_decision_change_pct':100*proposed.decision_change_rate_mean,
                            'status':'exploratory; accuracy profile has no validation harm guard' if profile=='accuracy' else 'exploratory; original empirical validation harm guards'})
    save_csv(OUTPUT/'notebook09_comparisons.csv',pd.DataFrame(comparisons))
    output={'code_version':CODE_VERSION,'study_signature':STUDY_SIGNATURE,'study_complete':ready,
        'configurations_complete':len(done),'configurations_required':15,'completed_results':done,
        'pending_configurations':[{k:r[k] for k in ['dataset','fold','seed']} for r in RESULTS if identity(r) not in {identity(d) for d in done}],
        'new_signal_model_training':0,'new_signal_model_inference':0,'fitted_meta_models_in_completed_runs':3*len(done),
        'meta_model_device':'CPU','original_models_or_policies_modified':False,
        'selection':'policy fitting; validation blend/margin selection; no stuck/test fitting or selection',
        'test_status':plan['test_status'],'primary_new_method':'gated_fusion_safe',
        'accuracy_profile_has_harm_guard':False,'harm_guarantee':False,'novelty_or_significance_claim':False,
        'old_single_candidate_oracle_is_upper_bound_for_fusion':False,
        'decision_change_rate_is_not_sensor_intervention_rate':True,'aggregation':plan['aggregation'],
        'output_directory':str(OUTPUT)}
    save_json(OUTPUT/'notebook09_run_report.json',output)
    files=[{'filename':p.name,'bytes':p.stat().st_size,'sha256':file_sha256(p)} for p in sorted(OUTPUT.glob('notebook09_*'))
           if p.is_file() and not p.name.endswith('.partial') and p.name!='notebook09_artifact_inventory.json']
    save_json(OUTPUT/'notebook09_artifact_inventory.json',files)
    if not final:
        print(f'Progress saved: {len(done)}/15 configurations.',flush=True);return
    print('\nNOTEBOOK 09 '+('COMPLETE' if ready else 'PAUSED — RERUN THE SAME CELL'),flush=True)
    print(f'Completed: {len(done)}/15 | Original signal models unchanged | Meta-models use CPU',flush=True)
    if len(aggregate):
        view=aggregate[aggregate.method.isin(['keep','harm_aware_policy','ternary_outcome_selector']+NEW_METHODS)&aggregate.scope.ne('clean')].copy()
        for m in ['macro_f1','harmful_rate','decision_change_rate']:view[m+'_pct']=100*view[m+'_mean']
        print(view[['dataset','scope','method','complete_seeds','macro_f1_pct','harmful_rate_pct','decision_change_rate_pct']]
              .to_string(index=False,float_format=lambda x:f'{x:.3f}'),flush=True)
    print('Accuracy profile may trade more harm for F1. Safe profile has empirical validation guards, not test guarantees.',flush=True)
    print('Previously inspected tests: all new comparisons are exploratory; no improvement or novelty is guaranteed.',flush=True)
    print('Outputs:',OUTPUT,flush=True)
    print('Share the final output and notebook09_aggregate_summary.csv, notebook09_comparisons.csv, notebook09_run_report.json.',flush=True)


def main():
    try:
        for result in RESULTS:
            if completed(result) is not None:continue
            check_budget();print('\nRUN',identity(result),flush=True)
            evaluate_job(result);publish();gc.collect()
    except PauseStudy as error:print(str(error),flush=True)
    publish(final=True)


main()


Mounted at /content/drive
NOTEBOOK 09 | conservative probability fusion | CPU | one executable cell
Both datasets; 15 configurations; cached signal-model predictions reused.
Three small meta-models per configuration. No CNN/reconstructor training or GPU needed.
Exploratory extension: existing test results are already known.
Safe profile retains the original validation harm guards. Accuracy profile has no harm guard.
Exploratory study plan frozen: cc39631c46bd6a5a

RUN dsads_fold_0_seed_2026
  global_fusion epoch 01/50 | policy loss 0.5482
  global_fusion epoch 10/50 | policy loss 0.5129
  global_fusion epoch 30/50 | policy loss 0.4780
  global_fusion epoch 50/50 | policy loss 0.4718
  gated_fusion epoch 01/50 | policy loss 0.5103
  gated_fusion epoch 10/50 | policy loss 0.2980
  gated_fusion epoch 30/50 | policy loss 0.2774
  gated_fusion epoch 50/50 | policy loss 0.2729
  residual_stacking epoch 01/50 | policy loss 0.2856
  residual_stacking epoch 10/50 | policy loss 0.1308
  residual